## STOR 893 - Find S2MPJ Problems

This is some helper code for the final project to find S2MPJ problems with certain attributes.

In [ ]:
import sys
import numpy as np
# Add S2MPJ and the python_problems subfolder to your path if it is not already configured
#myS2MPJdir = '/Your/Path/To/S2MPJ-main'
#sys.path.append(myS2MPJdir)
#sys.path.append(myS2MPJdir+'/python_problems')

myS2MPJdir = '/Users/moneill8/Documents/GitHub/S2MPJ'
sys.path.append(myS2MPJdir)
sys.path.append(myS2MPJdir+'/python_problems')

In [ ]:
fullproblist = []

with open(myS2MPJdir+"/list_of_python_problems", "r", encoding="utf-8") as file:
    for line in file:
        # rstrip('\n') removes the trailing newline character
        fullproblist.append(line.rstrip("\n"))
fullproblist

In [ ]:
import importlib

#Let's try to identify which problems have issues loading
#Many problems in S2MPJ through excepts on loading
skiplist = []

for problem_name in fullproblist:
    problem_name = problem_name.split('.')[0]
    problem_module = importlib.import_module(problem_name)
    problem_class = getattr(problem_module, problem_name)

    #Initialize the problem as prob
    try:
        prob = problem_class()
        print(prob.name)
    except Exception:
        skiplist.append(problem_name)
        print('Skip list added: ' + problem_name)

with open("skiplist.txt", "w", encoding="utf-8") as file:
    for toskip in skiplist:
        file.write(f"{toskip}\n")


In [ ]:
def checkconstantfunction(prob):
    # Returns true if the function is constant
    x0 = prob.x0
    numchecks = 100
    n = prob.n
    # Some problems don't even have an implemented objective function
    f0 = prob.fx(x0)
    if f0 is None:
        print("Skipping due to no objective function")
        return True
    for i in range(numchecks):
        x = x0 + np.random.randn(n)
        f = prob.fx(x)
        if(np.abs(f-f0) > 1**-10):
            #If f changes at least some
            return False

    return True

In [ ]:
# Run this to generate list from file instead
skiplist = []
with open("skiplist.txt","r") as file:
    for line in file:
        skiplist.append(line.rstrip())

In [ ]:
skiplist

In [ ]:
import importlib

maxdim=1000

#Let's try to identify equality constrained problems of a reasonable size
equalityprobs = []

for problem_name in fullproblist:
    problem_name = problem_name.split('.')[0]
    if(problem_name in skiplist):
        #Ignore entries in skiplist
        #print('Skipping ' + problem_name)
        continue
    problem_module = importlib.import_module(problem_name)
    problem_class = getattr(problem_module, problem_name)

    #Initialize the problem as prob
    try:
        prob = problem_class()
        if(not hasattr(prob, "neq") or prob.neq == 0):
            # No equality constraints
            continue
        if(prob.n + prob.neq > maxdim):
            # Skipping larger problems
            continue
        if(hasattr(prob, "nge") and hasattr(prob, "nle")):
            # Ignore inequality constrained problems
            if(prob.nge + prob.nle > 0):
                continue
        if(hasattr(prob, "xlower") and hasattr(prob, "xupper")):
            if(np.max(prob.xlower) > -np.inf or np.min(prob.xupper) < np.inf):
                # Ignore problems with bound constraints
                continue
        if(checkconstantfunction(prob)):
            # Ignore constant functions (feasibility problems)
            continue
        print(problem_name)
        equalityprobs.append(problem_name)
    except Exception as e:
        print('Issue with problem: ' + problem_name)
        print(type(e).__name__)
print(equalityprobs)

In [ ]:
with open("equalityprobs.txt", "w", encoding="utf-8") as file:
    for equalities in equalityprobs:
        file.write(f"{equalities}\n")

In [ ]:
# Run this to generate list from file instead
equalityprobs = []
with open("equalityprobs.txt","r") as file:
    for line in file:
        equalityprobs.append(line.rstrip())

In [ ]:
import importlib

maxdim=1000

#Let's try to identify inequality constrained problems of a reasonable size
equalityboundprobs = []
generalinequalityprobs = []

for problem_name in fullproblist:
    problem_name = problem_name.split('.')[0]
    if(problem_name in skiplist):
        #Ignore entries in skiplist
        continue
    problem_module = importlib.import_module(problem_name)
    problem_class = getattr(problem_module, problem_name)

    #Initialize the problem as prob
    try:
        print("Checking prob: " + problem_name)
        prob = problem_class()
        if(checkconstantfunction(prob)):
            # Ignore constant functions (feasibility problems)
            continue
        if(not hasattr(prob, "nge") and not hasattr(prob, "nle")):
            # No general inequalities, only looking for equalities + bounds in this case
            if(not hasattr(prob, "neq") or prob.neq == 0):
                # No equality constraints
                continue 
            if(hasattr(prob, "xlower") and hasattr(prob, "xupper")):
                if(np.max(prob.xlower) == -np.inf and np.min(prob.xupper) == np.inf):
                    # Ignore problems without bound constraints
                    continue
                print(np.max(prob.xlower))
            if(prob.n + prob.neq > maxdim):
                # Skipping larger problems
                continue
            print("Equality bound probs: " +  problem_name)
            equalityboundprobs.append(problem_name)
        else:
            nge = 0
            nle = 0
            if(hasattr(prob, "nge")):
                nge = prob.nge
            if(hasattr(prob, "nle")):
                nle = prob.nle
            if(nge+nle == 0):
                # No general inequalities, only looking for equalities + bounds in this case
                if(not hasattr(prob, "neq") or prob.neq == 0):
                    # No equality constraints
                    continue 
                if(hasattr(prob, "xlower") and hasattr(prob, "xupper")):
                    if(np.max(prob.xlower) == -np.inf and np.min(prob.xupper) == np.inf):
                        # Ignore problems without bound constraints
                        continue
                if(prob.n + prob.neq > maxdim):
                    # Skipping larger problems
                    continue
                print("Equality bound probs: " +  problem_name)
                equalityboundprobs.append(problem_name)
                continue
            if(prob.n + prob.nge + prob.nle > maxdim):
                # Skipping larger problems
                continue
            print("General ineq problem: " +  problem_name)
            generalinequalityprobs.append(problem_name)
    except Exception as e:
        print('Issue with problem: ' + problem_name)
        print(type(e).__name__)
print(equalityboundprobs)
print(generalinequalityprobs)

In [ ]:
with open("inequalityprobs.txt", "w", encoding="utf-8") as file:
    for ineq in generalinequalityprobs:
        file.write(f"{ineq}\n")

In [ ]:
with open("equalityboundsprobs.txt", "w", encoding="utf-8") as file:
    for eqb in equalityboundprobs:
        file.write(f"{eqb}\n")